# AIFlow — điều khiển huấn luyện và đổi giọng từ web
Chọn GPU runtime. Chỉ bật khi cần tải/xử lý/train/TTS. Dữ liệu nằm trên My Drive.
Tạo Colab Secrets `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, cho phép notebook đọc khi tải R2.
Sau khi kết nối, mọi thao tác chọn file, tải, duyệt dataset, train/resume, nghe mẫu, duyệt và đổi giọng thực hiện ở AIFlow `/voice-training`.
Không Run all: phần Batch và Dừng ở cuối chỉ chạy khi cần. Không chia sẻ output token.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import io, zipfile, os, sys, secrets, subprocess, json
uploaded = files.upload()  # aiflow-training-worker.zip từ AIFlow
if len(uploaded) != 1:
    raise ValueError('Chọn đúng một ZIP worker.')
root = Path('/content/aiflow-voice-control')
root.mkdir(exist_ok=True)
required = {'voice_training.py', 'train_resumable.py', 'training_control.py', 'audio_worker.py', 'youtube_source.py'}
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    if not required.issubset(archive.namelist()) or any(archive.getinfo(n).file_size > 2_000_000 for n in required):
        raise ValueError('Không đúng gói worker AIFlow.')
    for name in required:
        (root / name).write_bytes(archive.read(name))
del uploaded
drive.mount('/content/drive')
sys.path.insert(0, str(root))
os.chdir('/content')
os.environ['HF_HOME'] = '/content/drive/MyDrive/AIFlow/voice-training/hf-cache'
os.environ['AIFLOW_WORKER_STORAGE'] = '/content/drive/MyDrive/AIFlow/audio-worker'
os.environ['AIFLOW_WORKER_TOKEN'] = secrets.token_urlsafe(48)
import voice_training as training
training.run('apt-get', '-qq', 'update')
training.run('apt-get', '-qq', '-y', 'install', 'ffmpeg', 'espeak-ng', 'git')
if not training.UPSTREAM.exists():
    training.run('git', 'clone', 'https://github.com/pnnbao97/VieNeu-TTS.git', training.UPSTREAM)
training.run('git', '-C', training.UPSTREAM, 'checkout', '--detach', training.UPSTREAM_COMMIT)
training.run(sys.executable, '-m', 'pip', 'install', str(training.UPSTREAM) + '[finetune]',
             'boto3>=1.35,<2', 'yt-dlp>=2025.1,<2027', 'faster-whisper==1.1.1', 'av==18.0.0', 'ipywidgets>=8,<9',
             'fastapi>=0.115.12,<1', 'uvicorn>=0.34.2,<1', 'python-multipart>=0.0.12')
# LoRA uses ordinary FP32 weights; old optional torchao bundled by Colab breaks PEFT dispatch.
training.run(sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao')
training.gpu()
print('Chưa tải media, chưa load model. Bật API ở ô tiếp theo.')

In [ ]:
import threading, urllib.request, uvicorn, json
import audio_worker as worker
import training_control as control
if 'server' in globals() and not server.should_exit:
    raise RuntimeError('API đang chạy.')
worker.TOKEN = os.environ['AIFLOW_WORKER_TOKEN']
server = uvicorn.Server(uvicorn.Config(worker.app, host='127.0.0.1', port=8000, log_level='warning', access_log=False))
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o700)
tunnel_log = open('/content/aiflow-control-tunnel.log', 'w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT)

In [ ]:
import re, time
url = None
for _ in range(30):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/aiflow-control-tunnel.log').read_text())
    if match and server.started:
        url = match.group(0)
        break
    time.sleep(1)
if not url:
    raise RuntimeError('API/tunnel chưa sẵn sàng. Kiểm tra log rồi chạy lại.')
print('URL:', url)
print('TOKEN:', worker.TOKEN)
print('Dán vào AIFlow → Huấn luyện giọng → Kết nối điều khiển Colab.')

## Batch TTS không cần tunnel (tùy chọn)
Nếu không dùng API: chỉ chạy ô cài đặt đầu tiên, rồi hai ô sau. Chọn giọng đã duyệt từ Drive.
Dán JSON profile vào AIFlow → Colab & audio → Batch, xuất manifest tác vụ rồi upload ở ô tiếp theo.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
choices = training.saved_voices()
if not choices:
    raise RuntimeError('Chưa có giọng đã duyệt trên Drive.')
picker = widgets.Dropdown(options=choices, layout={'width': '95%'})
display(picker)

In [ ]:
if 'api_thread' in globals() and api_thread.is_alive():
    raise RuntimeError('Dừng API bằng ô cuối trước khi chạy batch không tunnel.')
import audio_worker as worker
if worker.ACTIVE or worker.CONTROL_BUSY:
    raise RuntimeError('Chờ tác vụ hiện tại hoàn tất.')
if worker.PIPELINE is not None:
    import training_control as control
    control.unload()
worker = training.configure_worker(picker.value)
print(json.dumps({'health': worker.health(), 'voices': worker.voices()}, ensure_ascii=False, indent=2))

In [ ]:
if 'api_thread' in globals() and api_thread.is_alive():
    raise RuntimeError('Dừng API bằng ô cuối trước khi chạy batch không tunnel.')
batch_upload = files.upload()
if len(batch_upload) != 1:
    raise ValueError('Upload một manifest JSON tác vụ audio từ AIFlow.')
manifest = json.loads(next(iter(batch_upload.values())))
if worker.ACTIVE or worker.CONTROL_BUSY:
    raise RuntimeError('Colab đang bận.')
output = Path('/content/aiflow-finetune-audio-results.zip')
worker.run_batch(manifest, output)
files.download(str(output))

## Dừng sau khi hoàn tất

In [ ]:
if worker.ACTIVE or worker.CONTROL_BUSY:
    raise RuntimeError('Chờ tác vụ hoặc yêu cầu dừng từ AIFlow trước.')
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate(); tunnel.wait(timeout=15)
if 'server' in globals():
    server.should_exit = True
    api_thread.join(timeout=10)
if 'tunnel_log' in globals():
    tunnel_log.close()
print('Chọn Runtime → Disconnect and delete runtime để ngừng dùng GPU.')